In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
data_path = "/content/drive/MyDrive/clinical_data"


In [ ]:
!pip install pandas faiss-cpu langchain openai transformers


In [ ]:
import pandas as pd

admissions = pd.read_csv(os.path.join(data_path, "ADMISSIONS.csv"))
patients = pd.read_csv(os.path.join(data_path, "PATIENTS.csv"))
labitems = pd.read_csv(os.path.join(data_path, "D_LABITEMS.csv"))
structured = pd.read_csv(os.path.join(data_path, "structured_medical_records.csv"))
labevents = pd.read_csv(os.path.join(data_path, "LABEVENTS.csv"), nrows=100000)  # sample for speed


In [ ]:
print("Structured columns:", structured.columns.tolist())
print("Admissions columns:", admissions.columns.tolist())


In [ ]:
def emr_to_text(struct_row):
    # Get admission info for this patient
    adm = admissions[admissions['subject_id'] == struct_row['subject_id']]
    if not adm.empty:
        diagnosis = adm['diagnosis'].values[0]
        insurance = adm['insurance'].values[0]
    else:
        diagnosis = "Unknown"
        insurance = "Unknown"

    return (
        f"Patient {struct_row['subject_id']} (Admission {struct_row['hadm_id']}) "
        f"Diagnosis: {diagnosis}. Insurance: {insurance}. "
        f"Report: {struct_row['medical_report']}"
    )

emr_texts = [emr_to_text(r) for _, r in structured.iterrows()]


In [ ]:
!pip install sentence-transformers faiss-cpu

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embeddings = model.encode(emr_texts, convert_to_numpy=True)

# Build FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)


In [ ]:
query = "Summarize abnormal lab results for patient 10076"
query_vec = model.encode([query])
distances, indices = index.search(query_vec, k=5)
for i in indices[0]:
    print(emr_texts[i])


In [ ]:
!pip install transformers sentencepiece


In [ ]:
!pip install --upgrade transformers


In [ ]:
!pip uninstall -y transformers sentence-transformers
!pip install transformers==4.41.2 sentence-transformers==3.0.1


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

retrieved_chunks = [
    "Patient 10076 has elevated creatinine levels indicating possible renal impairment.",
    "Hemoglobin is low at 9 g/dL, suggesting anemia.",
    "White blood cell count is high, consistent with infection."
]
context = " ".join(retrieved_chunks)

inputs = tokenizer("summarize: " + context, return_tensors="pt", max_length=1024, truncation=True)
summary_ids = model.generate(
    inputs["input_ids"],
    max_length=60,
    min_length=20,
    length_penalty=2.0,
    num_beams=4,
    early_stopping=True
)

print(tokenizer.decode(summary_ids[0], skip_special_tokens=True))


In [ ]:
!pip uninstall -y transformers sentence-transformers peft
!pip install transformers==4.41.2 sentence-transformers==3.0.1 peft==0.10.0


In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 1. Embedding model for retrieval
embedder = SentenceTransformer("all-MiniLM-L6-v2")

emr_texts = [
    "Patient 10076 has elevated creatinine levels indicating possible renal impairment.",
    "Hemoglobin is low at 9 g/dL, suggesting anemia.",
    "White blood cell count is high, consistent with infection."
]

corpus_embeddings = embedder.encode(emr_texts, convert_to_numpy=True)
index = faiss.IndexFlatL2(corpus_embeddings.shape[1])
index.add(corpus_embeddings)

# 2. Summarization model
model_name = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_name)
summarizer = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def summarize_patient(patient_id, k=2):
    query = f"Summarize abnormal lab results for patient {patient_id}"
    query_vec = embedder.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_vec, k)
    retrieved_chunks = [emr_texts[i] for i in indices[0]]
    context = " ".join(retrieved_chunks)

    inputs = tokenizer("summarize: " + context, return_tensors="pt", max_length=1024, truncation=True)
    summary_ids = summarizer.generate(
        inputs["input_ids"],
        max_length=80,
        min_length=20,
        length_penalty=2.0,
        num_beams=4,
        early_stopping=True
    )
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# Example usage
print(summarize_patient(10076))


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_name)
summarizer = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Define your EMR chunks
retrieved_chunks = [
    "Patient 10076 has elevated creatinine levels indicating possible renal impairment.",
    "Hemoglobin is low at 9 g/dL, suggesting anemia.",
    "White blood cell count is high, consistent with infection."
]

# Build the context string
context = " ".join(retrieved_chunks)

# Tokenize and summarize
inputs = tokenizer(context, return_tensors="pt", max_length=1024, truncation=True)
summary_ids = summarizer.generate(
    inputs["input_ids"],
    max_length=80,
    min_length=20,
    num_beams=6,
    length_penalty=2.0,
    early_stopping=True
)

print(tokenizer.decode(summary_ids[0], skip_special_tokens=True))


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_name)
summarizer = AutoModelForSeq2SeqLM.from_pretrained(model_name)

retrieved_chunks = [
    "Patient 10076 has elevated creatinine levels indicating possible renal impairment.",
    "Hemoglobin is low at 9 g/dL, suggesting anemia.",
    "White blood cell count is high, consistent with infection."
]

context = " ".join(retrieved_chunks)

inputs = tokenizer(context, return_tensors="pt", max_length=1024, truncation=True)
summary_ids = summarizer.generate(
    inputs["input_ids"],
    max_length=50,       # shorter target length
    min_length=10,       # enforce some compression
    num_beams=8,         # more beams for better quality
    length_penalty=2.0,  # penalize long outputs
    early_stopping=True
)

print(tokenizer.decode(summary_ids[0], skip_special_tokens=True))


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

context = "Patient 10076 has elevated creatinine levels indicating possible renal impairment. Hemoglobin is low at 9 g/dL, suggesting anemia. White blood cell count is high, consistent with infection."

inputs = tokenizer("summarize: " + context, return_tensors="pt", max_length=512, truncation=True)
summary_ids = model.generate(inputs["input_ids"], max_length=50, min_length=10, num_beams=4, early_stopping=True)

print(tokenizer.decode(summary_ids[0], skip_special_tokens=True))


In [ ]:
model_name = "t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


In [ ]:
summary_ids = model.generate(
    inputs["input_ids"],
    max_length=30,   # shorter target
    min_length=8,
    num_beams=6,
    length_penalty=3.0,  # strong penalty for long outputs
    early_stopping=True
)


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

context = """Patient 10076 has elevated creatinine levels indicating possible renal impairment.
Hemoglobin is low at 9 g/dL, suggesting anemia.
White blood cell count is high, consistent with infection."""

inputs = tokenizer("summarize: " + context, return_tensors="pt", max_length=512, truncation=True)
summary_ids = model.generate(inputs["input_ids"], max_length=30, min_length=8, num_beams=6, length_penalty=3.0, early_stopping=True)

print(tokenizer.decode(summary_ids[0], skip_special_tokens=True))


In [ ]:
model_name = "t5-large"


In [ ]:
summary_ids = model.generate(
    inputs["input_ids"],
    max_length=25,   # shorter target
    min_length=8,
    num_beams=8,
    length_penalty=3.5,  # strong penalty for long outputs
    early_stopping=True
)


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "t5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

context = """Patient 10076 has elevated creatinine levels indicating possible renal impairment.
Hemoglobin is low at 9 g/dL, suggesting anemia.
White blood cell count is high, consistent with infection."""

inputs = tokenizer("summarize: " + context, return_tensors="pt", max_length=512, truncation=True)
summary_ids = model.generate(inputs["input_ids"], max_length=25, min_length=8, num_beams=8, length_penalty=3.5, early_stopping=True)

print(tokenizer.decode(summary_ids[0], skip_special_tokens=True))


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "t5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

context = """Patient 10076 has elevated creatinine levels indicating possible renal impairment.
Hemoglobin is low at 9 g/dL, suggesting anemia.
White blood cell count is high, consistent with infection."""

inputs = tokenizer("summarize: " + context, return_tensors="pt", max_length=512, truncation=True)

summary_ids = model.generate(
    inputs["input_ids"],
    max_length=15,       # very short target
    min_length=5,
    num_beams=8,
    length_penalty=5.0,  # heavy penalty for long outputs
    early_stopping=True
)

print(tokenizer.decode(summary_ids[0], skip_special_tokens=True))


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load summarizer
model_name = "t5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
summarizer = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Example EMR corpus with patient IDs
emr_data = {
    10076: [
        "Elevated creatinine levels indicating possible renal impairment.",
        "Hemoglobin is low at 9 g/dL, suggesting anemia.",
        "White blood cell count is high, consistent with infection."
    ],
    10077: [
        "Normal creatinine but elevated liver enzymes."
    ],
    10078: [
        "Low platelet count and fever."
    ]
}

def summarize_patient(patient_id):
    # Join all notes for this patient
    context = " ".join(emr_data.get(patient_id, []))
    if not context:
        return "No records found."

    # Tokenize and summarize
    inputs = tokenizer("summarize: " + context, return_tensors="pt", max_length=512, truncation=True)
    summary_ids = summarizer.generate(
        inputs["input_ids"],
        max_length=20,
        min_length=5,
        num_beams=8,
        length_penalty=5.0,
        early_stopping=True
    )
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# Batch summarization
patient_ids = [10076, 10077, 10078]
summaries = {pid: summarize_patient(pid) for pid in patient_ids}

for pid, summary in summaries.items():
    print(f"Patient {pid}: {summary}")


In [ ]:
import csv

# Suppose you already have summaries in a dict
summaries = {
    10076: "renal impairment, anemia, infection",
    10077: "elevated liver enzymes",
    10078: "low platelets, fever"
}

# Save to CSV
with open("patient_summaries.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Patient_ID", "Summary"])
    for pid, summary in summaries.items():
        writer.writerow([pid, summary])

print("Summaries saved to patient_summaries.csv")


In [ ]:
# Example query
query = "What abnormalities does patient 10076 have?"

# Simple lookup
print(f"Answer: {summaries[10076]}")


In [ ]:
import matplotlib.pyplot as plt

labels = list(summaries.keys())
values = [len(summaries[pid].split(',')) for pid in labels]

plt.bar(labels, values)
plt.xlabel("Patient ID")
plt.ylabel("Number of abnormalities")
plt.title("Abnormalities per patient")
plt.show()


In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

# Suppose summaries is your dict {patient_id: summary}
all_abnormalities = []
for summary in summaries.values():
    for item in summary.split(','):
        all_abnormalities.append(item.strip().lower())

freqs = Counter(all_abnormalities)

plt.bar(freqs.keys(), freqs.values())
plt.xticks(rotation=45)
plt.ylabel("Number of patients")
plt.title("Abnormality frequency across patients")
plt.show()


In [ ]:
def find_patients_with(condition):
    return [pid for pid, summary in summaries.items() if condition.lower() in summary.lower()]

print(find_patients_with("anemia"))  # → [10076]


In [ ]:
import json
with open("patient_summaries.json", "w") as f:
    json.dump(summaries, f, indent=2)


In [ ]:
from sklearn.model_selection import train_test_split

# Example dataset
texts = [
    "Elevated creatinine, anemia, infection",
    "Normal creatinine, elevated liver enzymes",
    "Low platelets, fever"
]
labels = [1, 0, 1]  # Example binary labels

# Split into train, validation, test
X_train, X_temp, y_train, y_temp = train_test_split(texts, labels, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print("Train size:", len(X_train))
print("Validation size:", len(X_val))
print("Test size:", len(X_test))



In [ ]:
print("y_train:", y_train)
print("y_val:", y_val)
print("y_test:", y_test)


In [ ]:
texts = [
    "Elevated creatinine, anemia, infection",   # abnormal
    "Normal creatinine, elevated liver enzymes",# normal
    "Low platelets, fever",                     # abnormal
    "Anemia and infection present",             # abnormal
    "Liver enzymes elevated",                   # normal
    "Renal impairment detected",                # abnormal
    "Healthy patient record",                   # normal
    "No abnormalities found",                   # normal
    "Fever and anemia",                         # abnormal
    "Normal labs, no issues"                    # normal
]
labels = [1,0,1,1,0,1,0,0,1,0]  # 1 = abnormal, 0 = normal


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.3, random_state=42, stratify=labels
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

vectorizer = CountVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

clf = LogisticRegression()
clf.fit(X_train_vec, y_train)


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_pred = clf.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))


In [ ]:
# Step 1: Example dataset
texts = [
    "Elevated creatinine, anemia, infection",   # abnormal
    "Normal creatinine, elevated liver enzymes",# normal
    "Low platelets, fever",                     # abnormal
    "Anemia and infection present",             # abnormal
    "Liver enzymes elevated",                   # normal
    "Renal impairment detected",                # abnormal
    "Healthy patient record",                   # normal
    "No abnormalities found",                   # normal
    "Fever and anemia",                         # abnormal
    "Normal labs, no issues"                    # normal
]
labels = [1,0,1,1,0,1,0,0,1,0]  # 1 = abnormal, 0 = normal

# Step 2: Split train/test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.3, random_state=42, stratify=labels
)

# Step 3: Train model
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

vectorizer = CountVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

clf = LogisticRegression()
clf.fit(X_train_vec, y_train)

# Step 4: Evaluation
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_pred = clf.predict(X_test_vec)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

# Step 5: QA function
def ask_question(question, vectorizer, clf):
    q_vec = vectorizer.transform([question])
    pred = clf.predict(q_vec)[0]
    if pred == 1:
        return f"Answer: YES → {question} detected"
    else:
        return f"Answer: NO → {question} not detected"

# Step 6: Example questions
questions = [
    "Does patient 10076 have anemia?",
    "Does patient 10077 show infection?",
    "Which patients have renal impairment?",
    "Is fever present in patient 10078?",
    "Does patient record show normal labs?"
]

for q in questions:
    print(q, "=>", ask_question(q, vectorizer, clf))


In [ ]:
!ls /content


In [ ]:
import pandas as pd

df = pd.read_csv("/content/patient_summaries.csv")
df.head()


In [ ]:
!git config --global user.email "bhagojishashank@gmail.com"
!git config --global user.name "shashank-bhagoji"


In [ ]:
!git init

In [ ]:
!git add .

In [ ]:
!git commit -m "Added ML model and project files"

In [ ]:
!git remote add origin https://github.com/shashank-bhagoji/Clinical-QA-over-EMR-Data.git

In [ ]:
!git push https://REMOVED_TOKEN@github.com/shashank-bhagoji/Clinical-QA-over-EMR-Data.git main

In [ ]:
!git init

In [ ]:
!git add .

In [ ]:
!git commit -m "First commit"

In [ ]:
!git branch -M main

In [ ]:
!git remote add origin https://github.com/shashank-bhagoji/Clinical-QA-over-EMR-Data.git

In [ ]:
!git push -u https://REMOVED_TOKEN@github.com/shashank-bhagoji/Clinical-QA-over-EMR-Data.git main